# MarketScout AI — Analysis Debug Notebook

Tests the complete analysis pipeline **without** Docker, frontend, backend, OAuth, SSE, or Supabase.

## Two modes
| Mode | Description |
|------|-------------|
| **REAL** | Calls Tavily + Fireworks AI — requires API keys in `.env` |
| **MOCK** | Uses deterministic fixtures — no API cost, visibly labelled |

## How to run
1. `pip install -r agent-service/requirements.txt`
2. Copy `agent-service/.env.example` → `agent-service/.env` and fill in keys
3. `jupyter notebook notebooks/market_scout_analysis_debug.ipynb`
4. Edit **Cell 2** with your idea, then run all cells

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os
# Add agent-service to path so we can import agents/services directly
AGENT_SERVICE_ROOT = os.path.join(os.path.dirname(os.getcwd()), 'agent-service') \
    if os.path.basename(os.getcwd()) == 'notebooks' \
    else os.path.join(os.getcwd(), 'agent-service')
if AGENT_SERVICE_ROOT not in sys.path:
    sys.path.insert(0, AGENT_SERVICE_ROOT)

# Load .env from agent-service
from pathlib import Path
env_file = Path(AGENT_SERVICE_ROOT) / '.env'
if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv(env_file)
    print(f'Loaded .env from {env_file}')
else:
    print(f'WARNING: No .env found at {env_file}. Using environment variables or defaults.')

# Detect mode
FIREWORKS_KEY = os.getenv('FIREWORKS_API_KEY', '')
TAVILY_KEY    = os.getenv('TAVILY_API_KEY', '')
REAL_MODE     = bool(FIREWORKS_KEY and TAVILY_KEY)
mode_label    = '🟢 REAL MODE' if REAL_MODE else '🟡 MOCK MODE (synthetic data — NOT production)'
print(f'\n{mode_label}')
if not FIREWORKS_KEY:
    print('  ⚠  FIREWORKS_API_KEY not set — LLM calls will use test stub')
if not TAVILY_KEY:
    print('  ⚠  TAVILY_API_KEY not set — mock search results will be used')
print()

In [ ]:
# ── Cell 2: Enter your startup idea ──────────────────────────────────────────
# Edit these three variables and re-run all cells.

IDEA = """
An AI-powered platform that analyses soil health using satellite imagery and IoT sensors,
provides real-time fertilisation recommendations to farmers, and reduces crop waste by 30%.
"""

INDUSTRY         = "AgriTech"
HEALTHCARE_MODE  = False    # Set True for healthcare/medtech ideas

print(f'Idea:     {IDEA.strip()[:80]}...')
print(f'Industry: {INDUSTRY}')
print(f'Mode:     {mode_label}')

In [ ]:
# ── Cell 3: Run the pipeline ──────────────────────────────────────────────────
import asyncio, uuid, json
from IPython.display import display, HTML

job_id = str(uuid.uuid4())

# Register a dummy queue so the pipeline can push progress events
from orchestrator.pipeline import register_queue, run_pipeline
q = register_queue(job_id)

print(f'Starting pipeline (job_id={job_id[:8]}...)\n')

# Drain the queue asynchronously while running
async def _run_with_progress():
    pipeline_task = asyncio.create_task(
        run_pipeline(job_id=job_id, idea=IDEA.strip(),
                     industry=INDUSTRY, healthcare_mode=HEALTHCARE_MODE)
    )
    while not pipeline_task.done():
        try:
            event = await asyncio.wait_for(q.get(), timeout=0.5)
            pct    = event.get('progress', 0)
            agent  = event.get('current_agent', '')
            status = event.get('status', '')
            print(f'  [{pct:3d}%] {agent} — {status}')
        except asyncio.TimeoutError:
            pass
    return await pipeline_task

final_state = asyncio.run(_run_with_progress())
print(f'\nPipeline complete. Errors: {final_state.get("errors", [])}')

In [ ]:
# ── Cell 4: Agent output audit table ─────────────────────────────────────────
import pandas as pd

AGENT_KEYS = [
    ('idea_guard', 'Idea Guard'),
    ('research', 'Research'),
    ('competitors', 'Competitor'),
    ('scientific', 'Scientific'),
    ('patents', 'Patent'),
    ('funding', 'Funding'),
    ('trends', 'Trend'),
    ('research_gaps', 'Research Gap'),
    ('swot', 'SWOT'),
    ('opportunities', 'Opportunity'),
    ('risks', 'Risk'),
    ('innovation_score', 'Innovation Score'),
    ('validation', 'Validation'),
    ('strategy', 'Strategy'),
    ('report', 'Report'),
]

rows = []
for key, label in AGENT_KEYS:
    data = final_state.get(key) or {}
    completed  = '✓' if data and not data.get('parse_error') else '✗'
    eq         = data.get('evidence_quality', '—')
    sources    = len(data.get('sources') or [])
    vwarn      = len(data.get('_validation_warnings') or [])
    hflags     = len(data.get('_hallucination_flags') or [])
    # Score fields
    score_fields = ['novelty_score','market_saturation_score','funding_activity_score',
                    'research_maturity_score','patent_density_score','opportunity_score',
                    'innovation_score','risk_score','market_score']
    score_val  = next((data.get(f) for f in score_fields if data.get(f) is not None), '—')
    rows.append({
        'Agent': label, 'Completed': completed, 'EvidenceQuality': eq,
        'Score': score_val, 'Sources': sources,
        'ValWarnings': vwarn, 'HallucinationFlags': hflags,
    })

df = pd.DataFrame(rows)
display(df.to_html(index=False))

In [ ]:
# ── Cell 5: Score provenance table ────────────────────────────────────────────
innovation_data = final_state.get('innovation_score') or {}
breakdown       = innovation_data.get('score_breakdown') or []

if not breakdown:
    print('No score breakdown available.')
else:
    rows = []
    for d in breakdown:
        rows.append({
            'Dimension':    d['dimension'],
            'Producer':     d['source_agent'],
            'Raw Field':    d['source_field'],
            'Raw Value':    d['raw_value'] if d['raw_value'] is not None else 'NULL',
            'Multiplier':   d['evidence_multiplier'],
            'Adjusted':     d['adjusted_value'] if d['adjusted_value'] is not None else 'NULL',
            'Weight':       d['weight'],
            'Contribution': d['weighted_contribution'] if d['weighted_contribution'] is not None else 'NULL',
            'Status':       d['status'],
            'Warnings':     ' | '.join(d.get('warnings', [])) or '—',
        })
    df = pd.DataFrame(rows)
    display(df.to_html(index=False))

    final_score    = innovation_data.get('innovation_score')
    grade          = innovation_data.get('grade')
    coverage       = innovation_data.get('score_coverage', 0)
    is_provisional = innovation_data.get('is_provisional', False)

    print(f'\n━━━ FINAL SCORE: {final_score}/100  Grade: {grade}')
    print(f'    Coverage: {coverage:.0%}  |  Provisional: {is_provisional}')
    if innovation_data.get('consistency_warnings'):
        print('\n⚠ Consistency warnings:')
        for w in innovation_data['consistency_warnings']:
            print(f'    • {w}')

In [ ]:
# ── Cell 6: Consistency check results ────────────────────────────────────────
cc = final_state.get('consistency_check') or {}
passed   = cc.get('passed', True)
mock     = cc.get('mock_mode', False)
coverage = cc.get('score_coverage', 1.0)
errors   = cc.get('errors', [])
warnings = cc.get('warnings', [])

status_icon = '✅ PASS' if passed else '❌ FAIL'
print(f'Consistency check: {status_icon}')
print(f'Mock mode: {mock}  |  Score coverage: {coverage:.0%}')
if mock:
    display(HTML('<div style="background:#fff3cd;padding:10px;border-left:4px solid #ffc107">'
                 '<b>⚠ MOCK MODE ACTIVE</b> — Results are based on synthetic data. '
                 'Not suitable for real investment decisions.</div>'))
if errors:
    print('\n🔴 Errors:')
    for e in errors:
        print(f'  • {e}')
if warnings:
    print('\n🟡 Warnings:')
    for w in warnings:
        print(f'  • {w}')
if not errors and not warnings:
    print('No errors or warnings. 🎉')

In [ ]:
# ── Cell 7: Suspicious conditions detector ───────────────────────────────────
issues = []

# All scores identical
breakdown = (final_state.get('innovation_score') or {}).get('score_breakdown') or []
unique_vals = set(d['raw_value'] for d in breakdown
                  if d['status'] == 'available' and d['raw_value'] is not None
                  and d['dimension'] != 'competition_bonus')
if len(unique_vals) == 1:
    issues.append(f'ALL_SCORES_IDENTICAL: All sub-scores = {list(unique_vals)[0]}')

# Missing key agent outputs
for key in ['research','competitors','scientific','patents','funding','research_gaps']:
    if not final_state.get(key):
        issues.append(f'MISSING_AGENT_OUTPUT: {key} is None')

# Duplicate sources
all_urls = []
for key, _ in AGENT_KEYS:
    data = final_state.get(key) or {}
    all_urls.extend(data.get('sources') or [])
url_counts = {u: all_urls.count(u) for u in set(all_urls) if all_urls.count(u) > 1}
if url_counts:
    issues.append(f'DUPLICATE_SOURCES: {len(url_counts)} URL(s) used by multiple agents')

# No sources at all
if not all_urls:
    issues.append('NO_SOURCES: No web sources retrieved for any agent')

# Mock search active
if not TAVILY_KEY:
    issues.append('MOCK_SEARCH_ACTIVE: TAVILY_API_KEY not set — using synthetic data')

# Score fallback used
for d in breakdown:
    if d['status'] != 'available':
        issues.append(f'SCORE_STATUS_{d["status"].upper()}: {d["dimension"]} from {d["source_agent"]}')

print(f'Suspicious conditions detected: {len(issues)}')
for i in issues:
    print(f'  ⚠ {i}')
if not issues:
    print('  None detected ✅')

In [ ]:
# ── Cell 8: Quality Gate (PASS / FAIL) ───────────────────────────────────────
from config import settings

gate_checks = []

def gate(label, condition, required=True):
    status = '✅ PASS' if condition else ('❌ FAIL' if required else '⚠ WARN')
    gate_checks.append({'Check': label, 'Result': status})

innovation_data = final_state.get('innovation_score') or {}
cc_data         = final_state.get('consistency_check') or {}
breakdown       = innovation_data.get('score_breakdown') or []

# 1. No silent fallback score
no_fallback = all(d['status'] in ('available','missing') for d in breakdown)
gate('No silent fallback score (status ∈ {available, missing})', no_fallback)

# 2. No all-identical score pattern
gate('No all-identical score pattern', not cc_data.get('suspicious_identical_scores', False))

# 3. Score coverage ≥ 70%
gate('Score coverage ≥ 70%', innovation_data.get('score_coverage', 0) >= 0.70)

# 4. No critical consistency errors
gate('No critical consistency errors', len(cc_data.get('errors', [])) == 0)

# 5. Confidence breakdown exists
gate('Score breakdown populated', len(breakdown) > 0)

# 6. Mock mode disabled
gate('Mock search disabled (real Tavily data)', bool(settings.TAVILY_API_KEY), required=False)

# 7. Innovation score is not None
gate('Innovation score is not None', innovation_data.get('innovation_score') is not None)

# 8. Report agent did not re-invent scores
report_data  = final_state.get('report') or {}
canon_inno   = innovation_data.get('innovation_score')
report_score = report_data.get('market_score')
scores_match = (canon_inno is None or report_score is None or
                abs(float(canon_inno) - float(report_score)) <= 0)
gate('Report market_score == canonical innovation_score', scores_match)

# ── Display results
df = pd.DataFrame(gate_checks)
display(df.to_html(index=False))

fails = [r for r in gate_checks if 'FAIL' in r['Result']]
warns = [r for r in gate_checks if 'WARN' in r['Result']]

if not fails:
    display(HTML('<div style="background:#d4edda;padding:12px;border-left:4px solid #28a745">'
                 '<b>🎉 QUALITY GATE: PASS</b></div>'))
else:
    display(HTML(f'<div style="background:#f8d7da;padding:12px;border-left:4px solid #dc3545">'
                 f'<b>❌ QUALITY GATE: FAIL — {len(fails)} check(s) failed</b></div>'))
    for f in fails:
        print(f"  FAIL: {f['Check']}")

In [ ]:
# ── Cell 9: Raw JSON for any agent ───────────────────────────────────────────
# Change INSPECT to any key: 'research', 'competitors', 'innovation_score', etc.
INSPECT = 'innovation_score'

data = final_state.get(INSPECT)
if data:
    print(json.dumps(data, indent=2, default=str))
else:
    print(f'{INSPECT} is None or not in pipeline state.')

In [ ]:
# ── Cell 10: Generate PDF report (optional) ───────────────────────────────────
GENERATE_PDF = False   # Set True to generate and save the PDF

if GENERATE_PDF:
    from services.report_generator import generate_pdf_report
    pdf_bytes = generate_pdf_report(final_state)
    out_path  = Path(f'debug_report_{job_id[:8]}.pdf')
    out_path.write_bytes(pdf_bytes)
    print(f'PDF saved → {out_path.resolve()}  ({len(pdf_bytes):,} bytes)')
else:
    print('PDF generation skipped (set GENERATE_PDF = True to enable)')